# MONZA x jpt - Analise unica de resultados

Este notebook e o ponto unico para plotar tudo do experimento MONZA:

- FPR/FRR por round dos CCs disponiveis.
- Medias dos ultimos rounds.
- Metricas offline dos detectores DistilBERT e MLP.
- FPR/recall por tipo de ataque em `cc_type_results_*.csv`.
- Diagnostico especifico de `cc=10`/TargetLF em `cc_detail_results_10.csv`.

Ele aceita CSV antigo (`Round,FPR,FRR`) e CSV novo (`RunID,Round,FPR,FRR`). Quando existe `RunID`, usa o ultimo run completo; se o ultimo run estiver incompleto, cai para o run completo anterior.


In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

JPT = Path.cwd().resolve()
MONZA = JPT / 'PFLlibMonza' / 'system'
OUT = JPT
ANALYSIS_OUT = JPT / 'analysis_outputs'
ANALYSIS_OUT.mkdir(exist_ok=True)

EXPECTED_ROWS = int(os.environ.get('MONZA_EXPECTED_ROWS', '51'))
TAIL_ROUNDS = int(os.environ.get('MONZA_TAIL_ROUNDS', '30'))

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10

print('JPT:', JPT)
print('MONZA:', MONZA)
print('analysis_outputs:', ANALYSIS_OUT)
print('EXPECTED_ROWS:', EXPECTED_ROWS, '| TAIL_ROUNDS:', TAIL_ROUNDS)


## Helpers de carga

As funcoes abaixo escolhem o ultimo run completo quando existe `RunID`; para CSV legado, usam o ultimo trecho em que `Round` reiniciou.


In [ ]:
def savefig(name: str):
    root_path = OUT / name
    analysis_path = ANALYSIS_OUT / name
    plt.savefig(root_path, dpi=160, bbox_inches='tight')
    plt.savefig(analysis_path, dpi=160, bbox_inches='tight')
    return analysis_path


def latest_run_by_round_reset(df: pd.DataFrame, min_rows: int = 1) -> pd.DataFrame:
    rounds = df['Round'].astype(int).tolist()
    starts = [0]
    prev = rounds[0] if rounds else 0
    for idx, r in enumerate(rounds[1:], start=1):
        if r < prev:
            starts.append(idx)
        prev = r
    for start in reversed(starts):
        candidate = df.iloc[start:].reset_index(drop=True)
        if len(candidate) >= min_rows:
            return candidate
    return df.iloc[starts[-1]:].reset_index(drop=True) if starts else df.copy()


def latest_run(df: pd.DataFrame, min_rounds: int = 1) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    if 'RunID' in df.columns:
        run_ids = list(df['RunID'].drop_duplicates())
        for run_id in reversed(run_ids):
            candidate = df[df['RunID'] == run_id].reset_index(drop=True)
            if candidate['Round'].astype(int).nunique() >= min_rounds:
                return candidate
        return df[df['RunID'] == run_ids[-1]].reset_index(drop=True)
    return latest_run_by_round_reset(df, min_rows=min_rounds)


def load_fpr_frr_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    df = df.rename(columns={'False Positive Rate': 'FPR', 'False Rejection Rate': 'FRR'})
    required = {'Round', 'FPR', 'FRR'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'{path} sem colunas obrigatorias: {sorted(missing)}')
    keep = ['Round', 'FPR', 'FRR']
    if 'RunID' in df.columns:
        keep = ['RunID'] + keep
    df = df[keep].copy()
    df = df[df['Round'].astype(str) != 'Round'].reset_index(drop=True)
    df['Round'] = df['Round'].astype(int)
    df['FPR'] = df['FPR'].astype(float)
    df['FRR'] = df['FRR'].astype(float)
    return latest_run(df, min_rounds=EXPECTED_ROWS)


def load_json_if_exists(path: Path):
    if path.exists():
        with open(path) as f:
            return json.load(f)
    print(f'Arquivo ausente/ignorado: {path}')
    return None


def prefer_existing_dir(*candidates: Path) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]


## 1. FPR/FRR por round


In [ ]:
DEFENSES = {
    'cc=2 (cluster cosseno)': MONZA / 'fpr_frr_results_2.csv',
    'cc=3 (cosseno+score)': MONZA / 'fpr_frr_results_3.csv',
    'cc=6 (DistilBERT)': MONZA / 'fpr_frr_results_6.csv',
    'cc=7 (MLP+features)': MONZA / 'fpr_frr_results_7.csv',
    'cc=8 (MLP+validacao publica)': MONZA / 'fpr_frr_results_8.csv',
    'cc=9 (DistilBERT+MLP+label flip check)': MONZA / 'fpr_frr_results_9.csv',
    'cc=10 (DistilBERT+MLP+TargetLF)': MONZA / 'fpr_frr_results_10.csv',
}

COLORS = {
    'cc=2 (cluster cosseno)': '#1f77b4',
    'cc=3 (cosseno+score)': '#ff7f0e',
    'cc=6 (DistilBERT)': '#2ca02c',
    'cc=7 (MLP+features)': '#d62728',
    'cc=8 (MLP+validacao publica)': '#9467bd',
    'cc=9 (DistilBERT+MLP+label flip check)': '#8c564b',
    'cc=10 (DistilBERT+MLP+TargetLF)': '#17becf',
}

dfs = {}
missing = []
for name, path in DEFENSES.items():
    if not path.exists():
        missing.append(path.name)
        continue
    try:
        dfs[name] = load_fpr_frr_csv(path)
    except Exception as exc:
        print(f'Ignorando {path.name}: {exc}')

if not dfs:
    raise FileNotFoundError(f'Nenhum CSV fpr_frr_results_*.csv valido encontrado em {MONZA}')

for name, df in dfs.items():
    run_id = df['RunID'].iloc[0] if 'RunID' in df.columns else 'legacy'
    print(f'{name:45s} | run={run_id} | {len(df):3d} rows | FPR last={df.FPR.iloc[-1]:.4f} | FRR last={df.FRR.iloc[-1]:.4f}')
if missing:
    print('CSVs ausentes:', ', '.join(missing))


In [ ]:
rows = []
for name, df in dfs.items():
    tail = df.tail(TAIL_ROUNDS)
    rows.append({
        'Defesa': name,
        'Rows': len(df),
        'TailRows': len(tail),
        'FPR_mean': tail['FPR'].mean(),
        'FPR_std': tail['FPR'].std(),
        'FRR_mean': tail['FRR'].mean(),
        'FRR_std': tail['FRR'].std(),
        'Score (FPR+FRR)': tail['FPR'].mean() + tail['FRR'].mean(),
    })
summary = pd.DataFrame(rows).sort_values('Score (FPR+FRR)').reset_index(drop=True)
summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharex=True)
for name, df in dfs.items():
    color = COLORS.get(name, '#333333')
    axes[0].plot(df['Round'], df['FPR'], label=name, color=color, linewidth=2)
    axes[1].plot(df['Round'], df['FRR'], label=name, color=color, linewidth=2)
axes[0].set_title('False Positive Rate por round')
axes[0].set_xlabel('Round')
axes[0].set_ylabel('FPR')
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc='upper left', fontsize=8)
axes[0].set_ylim(-0.01, max(0.30, axes[0].get_ylim()[1]))
axes[1].set_title('False Rejection Rate por round')
axes[1].set_xlabel('Round')
axes[1].set_ylabel('FRR')
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc='upper left', fontsize=8)
axes[1].set_ylim(-0.01, max(0.60, axes[1].get_ylim()[1]))
plt.tight_layout()
savefig('plot_fpr_frr_by_round.png')


In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
for _, row in summary.iterrows():
    name = row['Defesa']
    ax.scatter(row['FPR_mean'], row['FRR_mean'], s=220, color=COLORS.get(name, '#333333'), edgecolors='black', linewidths=1.2, label=name)
    ax.annotate(name.split(' (')[0], (row['FPR_mean'], row['FRR_mean']), xytext=(10, 6), textcoords='offset points', fontsize=10, fontweight='bold')
max_fpr = max(float(summary['FPR_mean'].max()) * 1.25, 0.20)
max_frr = max(float(summary['FRR_mean'].max()) * 1.25, 0.35)
ax.set_title(f'Trade-off FPR x FRR - medias dos ultimos {TAIL_ROUNDS} rounds')
ax.set_xlabel('FPR medio')
ax.set_ylabel('FRR medio')
ax.set_xlim(-0.01, max_fpr)
ax.set_ylim(-0.01, max_frr)
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
savefig('plot_tradeoff_fpr_frr.png')


In [ ]:
summary_plot = summary.set_index('Defesa')[['FPR_mean', 'FRR_mean']]
fig, ax = plt.subplots(figsize=(12, 5))
summary_plot.plot.bar(ax=ax, color=['#3498db', '#e74c3c'], width=0.75)
ax.set_title(f'Defesas em producao FL: FPR vs FRR - medias dos ultimos {TAIL_ROUNDS} rounds')
ax.set_ylabel('Taxa')
ax.set_ylim(0, max(summary_plot.max().max() * 1.20, 0.30))
ax.set_xticklabels(summary_plot.index, rotation=20, ha='right')
ax.grid(alpha=0.3, axis='y')
for container in ax.containers:
    ax.bar_label(container, fmt='%.4f', fontsize=8, padding=2)
plt.tight_layout()
print('Ranking pelo criterio FPR+FRR (menor = melhor):')
print(summary[['Defesa', 'FPR_mean', 'FRR_mean', 'Score (FPR+FRR)']].to_string(index=False))
savefig('plot_summary_fpr_frr.png')


## 2. Metricas offline dos detectores


In [ ]:
BERT_DIR = prefer_existing_dir(JPT / 'detector_monza_cnn_mnist_v2', JPT / 'detector_monza_cnn_mnist')
MLP_DIR = prefer_existing_dir(JPT / 'detector_mlp_monza_cnn_mnist_v2', JPT / 'detector_mlp_monza_cnn_mnist')
print('BERT_DIR:', BERT_DIR)
print('MLP_DIR:', MLP_DIR)

bert_metrics = load_json_if_exists(BERT_DIR / 'metrics.json')
mlp_report = load_json_if_exists(MLP_DIR / 'report.json')

metric_rows = {}
if bert_metrics:
    default = bert_metrics.get('default_argmax', {})
    metric_rows['DistilBERT argmax'] = {k.capitalize(): default.get(k, np.nan) for k in ['accuracy', 'f1', 'precision', 'recall']}
    tuned = bert_metrics.get('tuned', {})
    if tuned:
        metric_rows['DistilBERT tuned'] = {k.capitalize(): tuned.get(k, np.nan) for k in ['accuracy', 'f1', 'precision', 'recall']}
if mlp_report:
    metrics = mlp_report.get('metrics', {})
    metric_rows['MLP argmax'] = {k.capitalize(): metrics.get(k, np.nan) for k in ['accuracy', 'f1', 'precision', 'recall']}
    tuned = mlp_report.get('tuned', {})
    if tuned:
        metric_rows['MLP tuned'] = {k.capitalize(): tuned.get(k, np.nan) for k in ['accuracy', 'f1', 'precision', 'recall']}
metrics_compare = pd.DataFrame(metric_rows).T
metrics_compare


In [ ]:
if metrics_compare.empty:
    print('Sem artefatos de detector para plotar. Rode o Passo 2 do HOWTO primeiro.')
else:
    fig, ax = plt.subplots(figsize=(11, 5))
    metrics_compare.plot.bar(ax=ax, width=0.72)
    ax.set_title('Detector metrics - eval set in-sample')
    ax.set_ylim(0, 1.05)
    ax.set_xticklabels(metrics_compare.index, rotation=15, ha='right')
    ax.grid(alpha=0.3, axis='y')
    for container in ax.containers:
        ax.bar_label(container, fmt='%.3f', fontsize=8, padding=3)
    plt.tight_layout()
    savefig('plot_detector_metrics.png')


In [ ]:
def by_type_rates_from_counts(by_type: dict) -> dict:
    values = {}
    for attack_type, item in (by_type or {}).items():
        total = item.get('total', 0)
        predicted = item.get('predicted_malicious', 0)
        rate = predicted / total if total else np.nan
        values['benign (FPR)' if attack_type == 'benign' else attack_type] = rate
    return values

by_type_sources = {}
if bert_metrics:
    bert_bt = by_type_rates_from_counts(bert_metrics.get('tuned', {}).get('by_type') or bert_metrics.get('default_argmax', {}).get('by_type', {}))
    if bert_bt:
        by_type_sources['DistilBERT'] = bert_bt
if mlp_report:
    mlp_bt = by_type_rates_from_counts(mlp_report.get('tuned', {}).get('by_type') or mlp_report.get('by_type', {}))
    if mlp_bt:
        by_type_sources['MLP features'] = mlp_bt

categories = ['benign (FPR)', 'malicious_zeros', 'malicious_random', 'malicious_shuffle', 'malicious_label']
by_type_df = pd.DataFrame({name: [values.get(cat, np.nan) for cat in categories] for name, values in by_type_sources.items()}, index=categories)
by_type_df


In [ ]:
if by_type_df.empty:
    print('Sem by_type dos detectores para plotar.')
else:
    fig, ax = plt.subplots(figsize=(11, 5))
    by_type_df.plot.bar(ax=ax, width=0.75)
    ax.set_title('FPR em benignos e recall por tipo de ataque - detectores offline')
    ax.set_ylabel('Taxa')
    ax.set_ylim(0, 1.1)
    ax.set_xticklabels(by_type_df.index, rotation=20, ha='right')
    ax.grid(alpha=0.3, axis='y')
    for container in ax.containers:
        ax.bar_label(container, fmt='%.2f', fontsize=8, padding=2)
    plt.tight_layout()
    savefig('plot_recall_by_attack.png')

    fig, ax = plt.subplots(figsize=(10, 2.8 + 0.25 * len(by_type_df)))
    ax.axis('off')
    table = ax.table(cellText=by_type_df.apply(lambda col: col.map(lambda x: '' if pd.isna(x) else f'{x:.3f}')).values, rowLabels=by_type_df.index, colLabels=by_type_df.columns, cellLoc='center', loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.35)
    ax.set_title('FPR/recall por tipo de ataque - detectores offline', pad=16)
    plt.tight_layout()
    savefig('plot_detector_attack_table.png')

    label_row = by_type_df.loc[['malicious_label']].T.rename(columns={'malicious_label': 'label_flip_recall'})
    fig, ax = plt.subplots(figsize=(7, 4))
    label_row.plot.bar(ax=ax, legend=False, color='#8c564b')
    ax.set_title('Recall em malicious_label - detectores offline')
    ax.set_ylabel('Recall')
    ax.set_ylim(0, 1.05)
    ax.set_xticklabels(label_row.index, rotation=15, ha='right')
    ax.grid(alpha=0.3, axis='y')
    for container in ax.containers:
        ax.bar_label(container, fmt='%.3f', fontsize=9, padding=3)
    plt.tight_layout()
    savefig('plot_label_flip_recall.png')


## 3. FPR/recall por tipo de ataque nos CCs

Usa `cc_type_results_*.csv` quando existirem. Estes arquivos sao gerados pelos CCs 6-10.


In [ ]:
def load_cc_type(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if df.empty:
        return df
    df.columns = [c.strip() for c in df.columns]
    required = {'Round', 'CC', 'AttackType', 'Total', 'Removed', 'Rate', 'Metric'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'{path.name} sem colunas: {sorted(missing)}')
    df = latest_run(df, min_rounds=min(TAIL_ROUNDS, EXPECTED_ROWS))
    for col in ['Round', 'CC', 'Total', 'Removed']:
        df[col] = df[col].astype(int)
    df['Rate'] = df['Rate'].astype(float)
    return df

cc_type_frames = []
for path in sorted(MONZA.glob('cc_type_results_*.csv')):
    try:
        df = load_cc_type(path)
        if not df.empty:
            cc_type_frames.append(df)
    except Exception as exc:
        print(f'Ignorando {path.name}: {exc}')

if cc_type_frames:
    cc_type = pd.concat(cc_type_frames, ignore_index=True)
    cc_type['Defense'] = 'cc=' + cc_type['CC'].astype(str)
else:
    cc_type = pd.DataFrame()
    print('Nenhum cc_type_results_*.csv encontrado ainda. Rode cc=6/7/8/9/10 para gerar.')
cc_type.head()


In [ ]:
if cc_type.empty:
    cc_summary = pd.DataFrame()
else:
    rows = []
    for cc, cc_group in cc_type.groupby('CC', sort=True):
        tail_round_values = sorted(cc_group['Round'].astype(int).unique())[-TAIL_ROUNDS:]
        tail_cc = cc_group[cc_group['Round'].isin(tail_round_values)]
        for attack_type, group in tail_cc.groupby('AttackType', sort=True):
            total = group['Total'].sum()
            removed = group['Removed'].sum()
            rows.append({
                'CC': cc,
                'Defense': f'cc={cc}',
                'AttackType': attack_type,
                'Total': int(total),
                'Removed': int(removed),
                'Rate': float(removed / total) if total else 0.0,
                'Metric': 'FPR_upload_benign' if attack_type == 'benign' else 'recall',
            })
    cc_summary = pd.DataFrame(rows)
cc_summary


In [ ]:
if cc_summary.empty:
    print('Sem resumo por tipo de ataque dos CCs para plotar.')
else:
    order = ['benign', 'malicious_label', 'malicious_zeros', 'malicious_random', 'malicious_shuffle']
    pivot = cc_summary.pivot_table(index='AttackType', columns='Defense', values='Rate', aggfunc='mean')
    pivot = pivot.reindex([x for x in order if x in pivot.index] + [x for x in pivot.index if x not in order])
    fig, ax = plt.subplots(figsize=(12, 5))
    pivot.plot.bar(ax=ax, width=0.78)
    ax.axhline(0.05, color='gray', linestyle=':', linewidth=1.3, label='FPR target 5%')
    ax.set_title('FPR em uploads benignos e recall por tipo de ataque nos CCs')
    ax.set_ylabel('Taxa')
    ax.set_xlabel('')
    ax.set_ylim(0, max(1.0, float(pivot.max().max()) * 1.15))
    ax.set_xticklabels(pivot.index, rotation=20, ha='right')
    ax.grid(axis='y', alpha=0.25)
    ax.legend()
    plt.tight_layout()
    savefig('plot_cc_recall_by_attack_type.png')

    label = cc_summary[cc_summary['AttackType'] == 'malicious_label'].sort_values('CC')
    if not label.empty:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(label['Defense'], label['Rate'], color='#8c564b')
        ax.set_title('Recall dos CCs em malicious_label')
        ax.set_ylabel('Recall')
        ax.set_ylim(0, max(1.0, float(label['Rate'].max()) * 1.2))
        ax.grid(axis='y', alpha=0.25)
        for idx, value in enumerate(label['Rate']):
            ax.text(idx, value + 0.02, f'{value:.2f}', ha='center', va='bottom')
        plt.tight_layout()
        savefig('plot_cc_malicious_label_recall.png')

    cc_summary.to_csv(ANALYSIS_OUT / 'cc_attack_type_summary.csv', index=False)
    cc_summary


## 4. Diagnostico TargetLF (`cc=10`)

Usa `cc_detail_results_10.csv` quando existir. Mostra score, gate comportamental e cap por tipo de ataque.


In [ ]:
detail_path = MONZA / 'cc_detail_results_10.csv'
if detail_path.exists():
    detail = pd.read_csv(detail_path)
    detail.columns = [c.strip() for c in detail.columns]
    detail = latest_run(detail, min_rounds=min(TAIL_ROUNDS, EXPECTED_ROWS))
    tail_rounds = sorted(detail['Round'].astype(int).unique())[-TAIL_ROUNDS:]
    detail_tail = detail[detail['Round'].astype(int).isin(tail_rounds)].copy()
else:
    detail = pd.DataFrame()
    detail_tail = pd.DataFrame()
    print('cc_detail_results_10.csv ausente. Rode cc=10 para gerar diagnostico TargetLF.')

detail_tail.head()


In [ ]:
if detail_tail.empty:
    target_summary = pd.DataFrame()
else:
    score_cols = [
        'MLPScore', 'BERTScore', 'TargetLFScore', 'TargetLFMarginDelta',
        'TargetLFLossDelta', 'TargetLFTargetProbDelta', 'TargetLFHeadScore', 'TargetLFFinalDelta',
    ]
    rows = []
    for attack_type, group in detail_tail.groupby('AttackType', sort=True):
        total = len(group)
        row = {
            'AttackType': attack_type,
            'Total': total,
            'Removed': int(group['Removed'].sum()) if 'Removed' in group else 0,
            'Rate': float(group['Removed'].sum() / total) if total and 'Removed' in group else 0.0,
            'TargetLFHitRate': float(group.get('TargetLFHit', pd.Series(0, index=group.index)).sum() / total) if total else 0.0,
            'TargetLFScoreOutlierRate': float(group.get('TargetLFScoreOutlier', pd.Series(0, index=group.index)).sum() / total) if total else 0.0,
            'TargetLFBehaviorAllowedRate': float(group.get('TargetLFBehaviorAllowed', pd.Series(0, index=group.index)).sum() / total) if total else 0.0,
            'TargetLFCappedRate': float(group.get('TargetLFCapped', pd.Series(0, index=group.index)).sum() / total) if total else 0.0,
        }
        for col in score_cols:
            if col in group:
                row[f'{col}_mean'] = float(group[col].mean())
                row[f'{col}_p95'] = float(group[col].quantile(0.95))
        rows.append(row)
    target_summary = pd.DataFrame(rows)
target_summary


In [ ]:
if target_summary.empty:
    print('Sem diagnostico TargetLF para plotar.')
else:
    plot_cols = ['Rate', 'TargetLFHitRate', 'TargetLFScoreOutlierRate', 'TargetLFBehaviorAllowedRate', 'TargetLFCappedRate']
    available = [c for c in plot_cols if c in target_summary.columns]
    fig, ax = plt.subplots(figsize=(12, 5))
    target_summary.set_index('AttackType')[available].plot.bar(ax=ax, width=0.78)
    ax.set_title('cc=10 TargetLF - remocao, gates e cap por tipo')
    ax.set_ylabel('Taxa')
    ax.set_ylim(0, 1.05)
    ax.set_xticklabels(target_summary['AttackType'], rotation=20, ha='right')
    ax.grid(axis='y', alpha=0.25)
    for container in ax.containers:
        ax.bar_label(container, fmt='%.2f', fontsize=7, padding=2)
    plt.tight_layout()
    savefig('plot_target_lf_gates_by_attack.png')

    score_plot_cols = [c for c in ['TargetLFScore_mean', 'TargetLFFinalDelta_mean', 'TargetLFMarginDelta_mean'] if c in target_summary.columns]
    if score_plot_cols:
        fig, ax = plt.subplots(figsize=(12, 5))
        target_summary.set_index('AttackType')[score_plot_cols].plot.bar(ax=ax, width=0.78)
        ax.set_title('cc=10 TargetLF - scores medios por tipo')
        ax.set_ylabel('Valor medio')
        ax.set_xticklabels(target_summary['AttackType'], rotation=20, ha='right')
        ax.grid(axis='y', alpha=0.25)
        plt.tight_layout()
        savefig('plot_target_lf_scores_by_attack.png')

    target_summary.to_csv(ANALYSIS_OUT / 'target_lf_summary.csv', index=False)
    target_summary


## Conclusoes rapidas

- Use `plot_summary_fpr_frr.png` e `plot_fpr_frr_by_round.png` para comparar defesas por round.
- Use `plot_cc_recall_by_attack_type.png` para comparar cada CC por tipo de ataque quando `cc_type_results_*.csv` existir.
- Use `plot_target_lf_gates_by_attack.png` e `plot_target_lf_scores_by_attack.png` para diagnosticar especificamente `malicious_label` no `cc=10`.
- Apos a canonizacao do preprocess, retreine o DistilBERT antes de comparar `cc=6/9/10`.
- Apos mudanca em `features.py`, retreine o MLP e confira `feature_names` no artefato.
